In [ ]:
## Nikolay Vorontsov, 23.11.2024, Prompting Dataset with hallucinations
## required files: label_training_data_with_llm_2.0.env
##                 mushroom.en-val.v2.jsonl

In [ ]:
#INSTALL DEPENDENCIES

!pip install google-generativeai

In [ ]:
# IMPORT LIBRARIES

import configparser
import google.generativeai as genai
import json

from google.colab import userdata

import random

from google.colab import drive
drive.mount('/content/drive')

import time

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# DEFINE VARIABLES

Your_API_Key = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=Your_API_Key)

model_used = "gemini-1.5-flash"
model = genai.GenerativeModel(model_name="gemini-1.5-flash")
prompts = configparser.ConfigParser()

prompts.read('/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/label_training_data_with_llm_2.0.env')

output_file="/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/label_training_data_with_llm_2.0_outputs/mushroom.en-train_nolable.v1.labelled_with_gemini1.5.jsonl"

training_set =  #'/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/mushroom.en-train_nolabel.v1.jsonl'

In [ ]:
# "a" creates the file if it doesn't exist
with open(output_file, "a") as file:
    pass  # Do nothing, just ensure the file exists

print(f"{output_file} is created or already exists.")

/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/label_training_data_with_llm_2.0_outputs/mushroom.en-train_nolable.v1.labelled_with_gemini1.5.jsonl is created or already exists.


In [ ]:
# Function to load the .jsonl file and parse its contents
def load_jsonl(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        # Read each line and parse as a JSON object
        return [json.loads(line) for line in file]

# Load the data
data = load_jsonl(training_set)


In [ ]:
#FUNCTIONS

In [ ]:
# COMPILE PROMPTS
def define_prompt(datapoint):

  #samples can be also imported from a jsonl file.
  Sample1 = prompts.get('SAMPLES', 'Sample1')

  prompt1 = (
      f"{prompts.get('PROMPTS', 'p0')}"
      f"{datapoint}"
      f"{prompts.get('PROMPTS', 'p1')}"
      f"{prompts.get('PROMPTS', 'p2')}"
      f"{prompts.get('PROMPTS', 'p3')}"
      f"{prompts.get('PROMPTS', 'p4')}"
      )

  return prompt1

In [ ]:
## TESTING define_prompt()
d = data[500]
print(d)
print()
print(define_prompt(d))

{'lang': 'EN', 'model_id': 'tiiuae/falcon-7b-instruct', 'model_input': 'Which Roman emperors were born in Lugdunum?', 'model_output_text': 'There are no Roman Emperorers born specifically in the city of Lugduum. However, some Roman rulers were known to have been born or raised in nearby cities such as Lyon and Toulouse.', 'model_output_logits': [-5.3361186981, -14.2025537491, -9.263584137, -9.6972675323, -3.2389011383, -10.7202615738, -13.9871044159, -14.2937202454, -8.9771575928, -12.0585670471, -9.1425609589, -7.2087049484, -9.908411026, -9.7210350037, -12.2368383408, -8.3032875061, -2.6280097961, -7.4861001968, -13.2629756927, -7.7696428299, -7.0299930573, -3.2880690098, -8.0829715729, -6.2061033249, -9.4560079575, -7.389585495, -5.1102995872, -6.3118700981, -9.1537303925, -7.3931570053, -11.2871236801, -6.7123560905, -4.2814245224, -10.1606140137, -10.6013240814, -6.82122612, -10.2073726654, -6.7649073601, -11.1015539169, -14.9314374924, -7.9172339439], 'model_output_tokens': ['The

In [ ]:
h = model.generate_content(define_prompt(d)).text.split("\n")
print(h)

['There are no Roman Emperorers born specifically in the city of Lugduum.', 'However, some Roman rulers were known to have been born or raised in nearby cities such as Lyon and Toulouse.', '']


In [ ]:
def process_data(datapointX):
    hallucinated_words = model.generate_content(define_prompt(datapointX)).text.split("\n")[:-1] #last element is "", remove it with [:-1]
    return hallucinated_words

In [ ]:
## TESTINS process_data() function
h = process_data(d)
print(h)

for word in h:
  print(word)

['There are no Roman Emperorers born specifically in the city of Lugduum', 'However, some Roman rulers were known to have been born or raised in nearby cities such as Lyon and Toulouse']
There are no Roman Emperorers born specifically in the city of Lugduum
However, some Roman rulers were known to have been born or raised in nearby cities such as Lyon and Toulouse


In [ ]:
def find_spans(datapointX, hallucinated_words):

  model_output_text = datapointX["model_output_text"]

  spans = []

  for word in hallucinated_words:
      # Initialize the starting index for each word search
      # the words can appear >1 times in text
      start_index = 0
      while True:
          start_index = model_output_text.find(word, start_index)
          if start_index == -1:
              break
          end_index = start_index + len(word)
          span = [start_index, end_index]
          spans.append(span)
          # Move the starting index past the current word to avoid overlapping results
          start_index = end_index

  return spans

In [ ]:
#TESTING find_spans() function
print(d["model_output_text"])
print(h)
result = find_spans(d, h)
print(result)

There are no Roman Emperorers born specifically in the city of Lugduum. However, some Roman rulers were known to have been born or raised in nearby cities such as Lyon and Toulouse.
['There are no Roman Emperorers born specifically in the city of Lugduum', 'However, some Roman rulers were known to have been born or raised in nearby cities such as Lyon and Toulouse']
[[0, 70], [72, 180]]


In [ ]:
def last_line_id(file_path):
  # Read the last line of the file
  last_line = None
  with open(file_path, "r") as file:
      for line in file:
          last_line = line.strip()  # Store the current line

  # Parse the JSON object from the last line
  if last_line:
      last_data = json.loads(last_line)
      #print("Last JSON object:", last_data)
      return last_data["id"]
  else:
      print("The file is empty")
      return None


In [ ]:
last_line_id(output_file)

The file is empty


In [ ]:
def label_and_save_data(data):
    processed_count = 0  # Counter for newly processed entries

    for id, datapoint in enumerate(data, start=1):
        # Get the last processed ID from the output file
        last_processed_id = last_line_id(output_file) or 0
        #print("Last processed ID:", last_processed_id)

        # Skip already processed entries
        if id <= last_processed_id:
            continue

        # Define prompt and process data
        prompt = define_prompt(datapoint)
        hallucinated_words = process_data(datapoint)

        # Save the datapoint to the JSONL file
        with open(output_file, "a", encoding='utf-8') as jsonl_file:
            datapoint_labelled = {
                "id": id,
                "llm-for-analysis": model_used,
                "lang": datapoint["lang"],
                "model_id": datapoint["model_id"],
                "model_input": datapoint["model_input"],
                "model_output_text": datapoint["model_output_text"],
                "model_output_logits": datapoint["model_output_logits"],
                "hallucinated_words": hallucinated_words,
                "hard_labels": find_spans(datapoint, hallucinated_words),
            }

            jsonl_file.write(json.dumps(datapoint_labelled) + "\n")
            print(datapoint_labelled["model_output_text"])
            print(datapoint_labelled["hallucinated_words"])
            print(datapoint_labelled["hard_labels"])

        # Increment the processed count
        processed_count += 1

        # Stop processing after 10 new entries
        if processed_count >= 10:
            print(f"Processed {processed_count} entries. Waiting for 30 sec. at ID {id}.")
            time.sleep(30)

            processed_count = 0


In [ ]:
label_and_save_data(data)

 The two sides are called the “Sectors” and the "Hegemony". 
['âĢľ', 'âĢĿ']
[]
 The two sides are called the “Sect” and the "Hive". 
['âĢľ', 'âĢĿ']
[]
 The two sides are called the “Sect” and the "Heretics". 
['âĢľ', 'âĢĿ']
[]
 The two sides are called the “Hegemony” and the "Anarchy". 
['âĢľ', 'âĢĿ']
[]
 The two sides are called the “Black Hand” and the "White Hand". 
['âĢľ', 'âĢĿ', '"', '"']
[[51, 52], [62, 63], [51, 52], [62, 63]]
 The two sides are called the “Sectors” and the "Races". 
['“Sectors”', '"Races"']
[[30, 39], [48, 55]]
 The two sides are called the “Consortium” and the "Anticonsortium". 
['âĢľ', 'âĢĿ', '"', '"']
[[51, 52], [66, 67], [51, 52], [66, 67]]
 Captain John Morgan was born in 1670. He was a pirate who raided Spanish ships in the Caribbean. 
['Captain John Morgan', '1670']
[[1, 20], [33, 37]]
 Captain Henry Morgan was the first person to sail under the name Captain John Morgan. He was born in 1635 and died in 1702. 
['Captain Henry Morgan', 'John Morgan', '1635

TooManyRequests: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: Resource has been exhausted (e.g. check quota).

In [ ]:
## Resave a copy with no extra keys.

with open(output_file, "r", encoding='utf-8') as jsonl_file:
    lines = jsonl_file.readlines()

    for line in lines:
        data_to_resave = json.loads(line)

        # Save the datapoint to the JSONL file
        with open(mushroom.en-val.v2.unlabeled.labelled_with_gemini1.5_no_extra_keys.jsonl, "a", encoding='utf-8') as jsonl_file:
            datapoint_labelled = {
                "id": data_to_resave["id"],
                "llm-for-analysis": model_used,
                "lang": datapoint["lang"],
                "model_id": datapoint["model_id"],
                "model_input": datapoint["model_input"],
                "model_output_text": datapoint["model_output_text"],
                "model_output_logits": datapoint["model_output_logits"],
                "hallucinated_words": hallucinated_words,
                "hard_labels": find_spans(datapoint, hallucinated_words),
            }
            jsonl_file.write(json.dumps(datapoint_labelled) + "\n")

